# How a small Vision Transformer classifies an image, and why it trails a ResNet-18 on CIFAR-10

**DS 677 Deep Learning, Fall 2026 (Model Review and Demo track).** Rishabh Verma.
Repository: [NJIT-DS677-Deep-Learning/DS677-project](https://github.com/NJIT-DS677-Deep-Learning/DS677-project).
[Open this notebook in Google Colab](https://colab.research.google.com/github/NJIT-DS677-Deep-Learning/DS677-project/blob/main/demo/demo.ipynb).

> **Claim (approved November 12, 2026).** We claim that a small Vision Transformer trained from scratch on CIFAR-10 stays at least 1 accuracy point below a ResNet-18 at every patch size we test (8x8, 4x4 and 2x2 pixels), and we will test it against a ResNet-18 baseline trained with the same data, augmentation and 300-epoch budget on CIFAR-10.

This notebook takes CIFAR-10 test images and follows them through the trained models until each model outputs a class. Every intermediate step of the Vision Transformer is computed by hand and checked against the library model, so nothing is a black box.

| Section | What it shows |
| --- | --- |
| 1. The input | What a CIFAR-10 image is and how it is prepared for the models |
| 2. ViT-Lite-7, step by step | Patches, patch embeddings, CLS token and position embeddings, the 7 transformer blocks, the classification head, and where the attention looks |
| 3. Patch size and the conv stem | The same image cut into 16, 64 or 256 tokens, and a small convolutional stem instead of patches (the perturbation study) |
| 4. The baseline: ResNet-18 | The same image through convolution stages, pooling and a linear head |
| 5. Replication anchor and claim check | Test-set accuracy of every model, against the training runs, the published numbers and the claim's 1-point margin |
| 6. Try your own image | Any picture, resized to 32x32, through every model |

**Models.** Every model was trained on Wulver with the recipe in `experiments/configs/`. The weights loaded here are seed 0 of each run, stored in `demo/weights/` in float16 (about 52 MB in total). The report quotes the mean of 3 seeds; section 5 shows both.

**Running it.** Run the cells from top to bottom; a GPU is optional. On a CPU, section 5 uses the first 2,000 test images instead of all 10,000 unless you change `N_EVAL`. The copy of this notebook in the repository was executed on Wulver (A100 slice, full test set) by `experiments/wulver/demo.sbatch`.

## 0. Setup

The first cell finds the repository (or clones it, e.g. on Colab) and makes sure the installed `transformers` has the same major version as the one the weights were saved with: the names of the layers inside the ViT changed between versions 4 and 5.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

REPO_URL = "https://github.com/NJIT-DS677-Deep-Learning/DS677-project.git"
here = Path.cwd().resolve()
REPO = next((p for p in [here, *here.parents] if (p / "src" / "models.py").exists()), None)
if REPO is None:  # running outside the repository (e.g. Google Colab): clone it next to the notebook
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    REPO = here / "DS677-project"
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

manifest = json.loads((REPO / "demo/weights/manifest.json").read_text())
need = manifest["transformers_version"]
have = version("transformers") if importlib.util.find_spec("transformers") else None
if have is None or have.split(".")[0] != need.split(".")[0]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"transformers=={need}"], check=True)
for package, module in [("safetensors", "safetensors"), ("pyyaml", "yaml"), ("pandas", "pandas")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
print("repository:", REPO)
print(f"weights: seed {manifest['seed']}, saved with transformers {need}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from safetensors.torch import load_file

from data import build_transform, load_split
from models import build_model

transformers.logging.set_verbosity_error()
torch.set_grad_enabled(False)  # inference only: nothing in this notebook trains
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else None  # as in training
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
N_EVAL = None  # section 5: None means all 10,000 test images on a GPU and the first 2,000 on a CPU
pd.set_option("display.precision", 2)
print(f"torch {torch.__version__}, transformers {transformers.__version__}, device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))

## 1. The input: a 32x32 colour image

CIFAR-10 has 60,000 colour images of 32x32 pixels in 10 classes: 50,000 for training and 10,000 for testing. Everything below uses the test images. No model saw them during training, and they were not used to choose any setting either (learning rates were chosen on a validation split carved out of the training images; see `experiments/decisions.md`).

On Wulver the images are already in `data/`. Elsewhere, the next cell downloads CIFAR-10 from the Hugging Face Hub (`uoft-cs/cifar10`) and keeps the test split, stored the same way as on Wulver: one `uint8` array of images and one array of integer labels.

In [ ]:
DATA = REPO / "data"
if not (DATA / "cifar10_test.npz").exists():
    if importlib.util.find_spec("datasets") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset

    ds = load_dataset("uoft-cs/cifar10", split="test")
    col = "img" if "img" in ds.column_names else "image"
    DATA.mkdir(exist_ok=True)
    np.savez(DATA / "cifar10_test.npz", x=np.stack([np.asarray(im.convert("RGB"), dtype=np.uint8) for im in ds[col]]),
             y=np.asarray(ds["label"], dtype=np.int64))

x_test, y_test = load_split(str(DATA), "test")  # images (N, 3, 32, 32) uint8, labels (N,)
print("images:", tuple(x_test.shape), x_test.dtype, "| labels:", tuple(y_test.shape), y_test.dtype)
print("test images per class:", dict(zip(CLASSES, torch.bincount(y_test).tolist())))

first_of_class = [int((y_test == c).nonzero()[0]) for c in range(10)]
fig, axes = plt.subplots(1, 10, figsize=(13, 1.9))
for ax, i in zip(axes, first_of_class):
    ax.imshow(x_test[i].permute(1, 2, 0).numpy(), interpolation="nearest")
    ax.set_title(f"{CLASSES[y_test[i]]}\ntest image {i}", fontsize=9)
    ax.axis("off")
plt.show()

**Preparing an image for the models.** The models never see raw 0 to 255 values. At test time every from-scratch model applies the same two steps, with no augmentation: scale the pixels to [0, 1], then subtract the per-channel mean and divide by the per-channel standard deviation of the CIFAR-10 training set. The result is a float tensor of shape (3, 32, 32); a batch of one image has shape (1, 3, 32, 32).

We follow test image `IMG` (image 0, a cat, by default) through every model. Change `IMG` to follow another one.

In [ ]:
WEIGHTS = REPO / "demo" / "weights"
RUNS = manifest["runs"]
eval_tf = build_transform(RUNS["vitlite_p4"]["config"]["data"], train=False)  # identical for every from-scratch model
print(eval_tf)

IMG = 0
img, label = x_test[IMG], int(y_test[IMG])
x = eval_tf(img).unsqueeze(0).to(DEVICE)
print(f"\ntest image {IMG}, true class: {CLASSES[label]}")
print(f"raw image:   shape {tuple(img.shape)}, {img.dtype}, values {img.min().item()} to {img.max().item()}")
print(f"model input: shape {tuple(x.shape)}, {x.dtype}, values {x.min().item():.2f} to {x.max().item():.2f}")

## 2. ViT-Lite-7, step by step (4x4 patches)

ViT-Lite-7 is the small Vision Transformer of Hassani et al. (2021), *Escaping the Big Data Paradigm with Compact Transformers*: the architecture of the original ViT (Dosovitskiy et al., 2021) shrunk for 32x32 images, with 7 transformer blocks, 4 attention heads, token width 256 and MLP width 512. Here it is built with the Hugging Face `ViTForImageClassification` class and a small configuration (`src/models.py`). With 4x4 patches the image goes through these steps:

| Step | Operation | Shape after the step |
| --- | --- | --- |
| 1 | Cut the image into 8x8 = 64 patches of 4x4 pixels x 3 colours | (1, 64, 48) |
| 2 | Map each patch to a 256-number token with one learned linear map | (1, 64, 256) |
| 3 | Prepend a learned CLS token and add a learned position embedding to every token | (1, 65, 256) |
| 4 | 7 transformer blocks: self-attention between all 65 tokens, then an MLP on each token | (1, 65, 256) |
| 5 | LayerNorm, keep only the CLS token, linear map to 10 class scores (logits), softmax | (1, 10) |

The next cell loads the trained weights and counts the parameters in each part.

In [ ]:
def load_model(run, attn="eager"):
    # Rebuild a model from its training config and load the exported seed-0 weights (float16 -> float32).
    cfg = RUNS[run]["config"]
    model = build_model({**cfg["model"], "attn": attn})
    state = load_file(str(WEIGHTS / RUNS[run]["file"]))
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in state.items()})
    return model.to(DEVICE).eval()


vit = load_model("vitlite_p4")  # "eager" attention so the attention weights can be read out
hf_vit = vit.hf.vit             # the Hugging Face ViTModel inside the classifier
blocks = hf_vit.layers if hasattr(hf_vit, "layers") else hf_vit.encoder.layer
emb = hf_vit.embeddings

n_params = lambda m: sum(p.numel() for p in m.parameters())
parts = [("patch embedding (step 2)", n_params(emb.patch_embeddings)),
         ("CLS token + position embeddings (step 3)", emb.cls_token.numel() + emb.position_embeddings.numel()),
         ("7 transformer blocks (step 4)", n_params(blocks)),
         ("final LayerNorm + classifier (step 5)", n_params(hf_vit.layernorm) + n_params(vit.hf.classifier))]
display(pd.DataFrame(parts, columns=["part", "parameters"]).set_index("part"))
print(f"total: {n_params(vit):,} parameters")

### Step 1: cut the image into patches

`unfold` cuts the (3, 32, 32) image into non-overlapping 4x4 squares, read row by row. Each patch is flattened into 3 x 4 x 4 = 48 numbers. The figure shows the same image cut with the three patch sizes of the perturbation study: smaller patches give more tokens, and attention cost grows with the square of the token count.

In [ ]:
p = 4
patches = x.unfold(2, p, p).unfold(3, p, p)                          # (1, 3, 8, 8, 4, 4)
patches = patches.permute(0, 2, 3, 1, 4, 5).reshape(1, -1, 3 * p * p)  # (1, 64, 48), patches row by row
print("patches:", tuple(patches.shape))

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.8))
for ax, ps in zip(axes, [8, 4, 2]):
    ax.imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest", extent=(0, 32, 32, 0))
    for t in range(0, 33, ps):
        ax.axhline(t, color="white", lw=0.7)
        ax.axvline(t, color="white", lw=0.7)
    n = (32 // ps) ** 2
    ax.set_title(f"{ps}x{ps} patches: {n} tokens\nattention matrix {n + 1}x{n + 1} per head", fontsize=10)
    ax.axis("off")
plt.show()

### Step 2: embed each patch

A single learned matrix maps every 48-number patch to a 256-number token. The library implements this as a convolution whose kernel size and stride both equal the patch size, which is the same operation. The cell computes it as a plain matrix product and checks that the result matches the library's patch embedding.

In [ ]:
proj = emb.patch_embeddings.projection          # Conv2d(3, 256, kernel_size=4, stride=4)
W = proj.weight.reshape(proj.out_channels, -1)  # (256, 48): one row per output number
tokens_by_hand = patches @ W.T + proj.bias      # (1, 64, 256)
tokens_library = emb.patch_embeddings(x)
print(proj)
print("patch tokens:", tuple(tokens_by_hand.shape),
      f"| largest difference from the library: {(tokens_by_hand - tokens_library).abs().max().item():.1e}")

### Step 3: CLS token and position embeddings

Self-attention treats its input as an unordered set, so the model would not know where a patch came from. ViT therefore adds a learned 256-number **position embedding** to the token at each of the 64 positions. It also prepends one extra learned token, the **CLS token**, which carries no image content at the start; after the last block, the classifier reads only this token.

In [ ]:
h0_by_hand = torch.cat([emb.cls_token.expand(1, -1, -1), tokens_by_hand], dim=1) + emb.position_embeddings
h0_library = emb(x)
print("tokens entering block 1:", tuple(h0_by_hand.shape),
      f"| largest difference from the library: {(h0_by_hand - h0_library).abs().max().item():.1e}")

# Nothing told the model that the 64 positions form an 8x8 grid. If it learned that from the data, each
# position's embedding should be most similar to its own neighbours.
pos = emb.position_embeddings[0, 1:]                                                 # (64, 256)
sim = torch.nn.functional.cosine_similarity(pos[:, None], pos[None], dim=-1).cpu()  # (64, 64)
fig, axes = plt.subplots(8, 8, figsize=(6.2, 6.2))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(sim[i].reshape(8, 8), cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.6, label="cosine similarity")
fig.suptitle("Similarity of each position's embedding (one small map per position)\n"
             "to the embeddings of all 64 positions", fontsize=10)
plt.show()

### Step 4: the transformer blocks

Each of the 7 blocks applies two sub-layers, each with LayerNorm before it and a residual connection around it:

1. **Multi-head self-attention.** Every token is mapped to a query, a key and a value (three learned 256 x 256 maps, split into 4 heads of 64 numbers). In each head, the attention matrix is softmax(Q K<sup>T</sup> / &radic;64): a 65 x 65 table whose row *i* says how much token *i* takes from each token *j*. The 4 heads' outputs are concatenated and mixed by one more 256 x 256 map. This is the only place where tokens exchange information.
2. **MLP**, applied to each token separately: 256 &rarr; 512, GELU, 512 &rarr; 256.

`block_by_hand` writes one block out with these operations, using the trained weights of the library's block. The cell runs all 7 blocks by hand and compares each block's output with the library's.

In [ ]:
def block_by_hand(block, h, n_heads=4):
    # One pre-norm transformer block written out (layer names of transformers 5.x).
    B, T, D = h.shape
    a = block.attention
    z = block.layernorm_before(h)
    split = lambda t: t.view(B, T, n_heads, D // n_heads).transpose(1, 2)            # (B, heads, T, 64)
    q, k, v = split(a.q_proj(z)), split(a.k_proj(z)), split(a.v_proj(z))
    att = torch.softmax(q @ k.transpose(-1, -2) / (D // n_heads) ** 0.5, dim=-1)       # (B, heads, T, T)
    h = h + a.o_proj((att @ v).transpose(1, 2).reshape(B, T, D))                        # attention + residual
    h = h + block.mlp.fc2(block.mlp.activation_fn(block.mlp.fc1(block.layernorm_after(h))))  # MLP + residual
    return h, att


out = vit.hf(pixel_values=x, output_attentions=True, output_hidden_states=True)
h, attentions = h0_by_hand, []
for i, block in enumerate(blocks, start=1):
    h, att = block_by_hand(block, h)
    attentions.append(att)
    print(f"block {i}: output {tuple(h.shape)}, attention {tuple(att.shape)}, largest difference from the "
          f"library: output {(h - out.hidden_states[i]).abs().max().item():.1e}, "
          f"attention {(att - out.attentions[i - 1]).abs().max().item():.1e}")

### Step 5: from the CLS token to a class

After the last block, a final LayerNorm is applied and only the CLS token (token 0) is kept. One linear map turns its 256 numbers into 10 scores, the **logits**, one per class. Softmax turns the logits into probabilities, and the predicted class is the one with the highest probability.

In [ ]:
cls = hf_vit.layernorm(h)[:, 0]              # (1, 256)
logits_by_hand = vit.hf.classifier(cls)      # (1, 10)
logits_library = vit(x)                      # the whole model in one call
probs = logits_by_hand.softmax(dim=-1)[0].cpu()
print("logits:", tuple(logits_by_hand.shape),
      f"| largest difference from the library: {(logits_by_hand - logits_library).abs().max().item():.1e}")
print(f"prediction: {CLASSES[int(probs.argmax())]} ({probs.max().item():.1%}); true class: {CLASSES[label]}")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 2.8), gridspec_kw={"width_ratios": [1, 3]})
ax0.imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest")
ax0.set_title(f"input: {CLASSES[label]}", fontsize=10)
ax0.axis("off")
ax1.barh(CLASSES[::-1], probs.numpy()[::-1], color=["#1baf7a" if c == label else "#b5b4ae" for c in range(10)][::-1])
ax1.set_xlim(0, 1)
ax1.set_xlabel("probability (ViT-Lite-7/4); the true class is in green")
for side in ("top", "right"):
    ax1.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

### Where the attention looks

The top row shows, for each block, how much the CLS token attends to each of the 64 patches (averaged over the 4 heads), drawn over the image. The table below it measures, for the attention between patches, the average distance in pixels between a patch and the patches it attends to: this image's value, and the value over 1,000 test images from the layer analysis in `results/analysis/`. If attention were spread evenly over the image, that distance would be about 16.5 pixels.

In [ ]:
grid = 8
fig, axes = plt.subplots(1, 7, figsize=(14, 2.5))
for i, (ax, att) in enumerate(zip(axes, attentions), start=1):
    cls_to_patches = att[0, :, 0, 1:].mean(0).reshape(grid, grid).cpu()
    ax.imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest", extent=(0, 32, 32, 0))
    ax.imshow(cls_to_patches, cmap="magma", alpha=0.6, extent=(0, 32, 32, 0), interpolation="nearest")
    ax.set_title(f"block {i}", fontsize=10)
    ax.axis("off")
fig.suptitle("Attention from the CLS token to the 64 patches, mean over heads (brighter = more attention)", fontsize=10)
plt.show()

# Mean attention distance between patches, in CIFAR pixels, as in src/analysis/attention_distance.py
idx = torch.arange(grid, dtype=torch.float32)
centres = torch.stack(torch.meshgrid(idx, idx, indexing="ij"), -1).reshape(-1, 2) * (32 / grid) + 32 / grid / 2
dist = torch.cdist(centres, centres).to(DEVICE)                                        # (64, 64)
this_image = []
for att in attentions:
    a = att[0, :, 1:, 1:]
    a = a / a.sum(-1, keepdim=True)                                                    # renormalised without CLS
    this_image.append((a * dist).sum(-1).mean(-1).cpu())                               # one value per head
table = pd.DataFrame({"block": range(1, 8),
                      "this image, mean over heads (px)": [d.mean().item() for d in this_image],
                      "this image, most local head (px)": [d.min().item() for d in this_image]})
analysis_file = REPO / "results/analysis/attention_distance.json"
if analysis_file.exists():
    ad = json.loads(analysis_file.read_text())
    seed0 = np.array(ad["models"]["vitlite_p4"]["seeds"]["0"]) * 32                    # fraction of width -> px
    table[f"{ad['n_images']} test images, mean over heads (px)"] = seed0.mean(1)
display(table.set_index("block"))

## 3. Patch size and the conv stem (the perturbation study)

The perturbation study changes one thing, the **tokenizer** that turns the image into tokens, and keeps everything else fixed: the same 7 blocks, data, augmentation, optimizer settings and 300 epochs.

* **Patch size 8, 4 or 2.** 16, 64 or 256 tokens of 8x8, 4x4 or 2x2 pixels. More tokens give finer detail but cost more: attention compares every token with every other.
* **Conv stem (extension).** Two stages of (3x3 convolution, ReLU, 3x3 max-pool with stride 2) replace the linear patch embedding. They produce an 8x8 grid of 64 tokens, the same count as 4x4 patches, but each token now summarises a 13x13-pixel window and neighbouring windows overlap, so the tokens carry the kind of local features a CNN computes in its first layers.

The cell loads all four ViTs, shows the cost of each tokenizer, and classifies the same image with each one.

In [ ]:
vits = {"vitlite_p8": "ViT-Lite-7/8", "vitlite_p4": "ViT-Lite-7/4", "vitlite_p2": "ViT-Lite-7/2",
        "vitlite_convstem": "ViT-Lite-7 + conv stem"}
models = {run: load_model(run) for run in vits}
cost_file = REPO / "results/model_cost.json"
cost = {r["model"]: r for r in json.loads(cost_file.read_text())} if cost_file.exists() else {}
rows = []
for run, name in vits.items():
    m = models[run]
    tokens = m.hf.vit.embeddings.position_embeddings.shape[1]
    pr = m(x).softmax(-1)[0].cpu()
    rows.append({"model": name, "tokens (with CLS)": tokens, "parameters": n_params(m),
                 "forward GFLOPs per image": cost.get(name, {}).get("forward_GFLOPs"),
                 "prediction": CLASSES[int(pr.argmax())], "probability": float(pr.max()),
                 f"probability of '{CLASSES[label]}'": float(pr[label])})
display(pd.DataFrame(rows).set_index("model"))

**Inside the conv stem.** The next cell runs the stem's layers one at a time and prints the shapes, then draws what one token sees: a 4x4 patch for ViT-Lite-7/4 against the 13x13 receptive field of the conv-stem token at the same grid position.

In [ ]:
stem = models["vitlite_convstem"].hf.vit.embeddings.patch_embeddings
s1 = stem.stage1(x)                          # 3x3 conv (3 -> 64 channels), ReLU, max-pool stride 2
s2 = torch.relu(stem.projection(s1))         # 3x3 conv (64 -> 256 channels), ReLU
s3 = stem.pool2(s2)                          # max-pool stride 2
tokens = s3.flatten(2).transpose(1, 2)       # 8x8 positions -> 64 tokens of 256 numbers
for name, t in [("input", x), ("after stage 1", s1), ("after the second conv", s2), ("after the second pool", s3),
                ("tokens", tokens)]:
    print(f"{name:24s}{tuple(t.shape)}")
print(f"largest difference from the library's stem: {(tokens - stem(x)).abs().max().item():.1e}")

r, c = 3, 3  # grid position of the token drawn below
fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
for ax, (title, x0, y0, size) in zip(axes, [("ViT-Lite-7/4: a 4x4 patch", 4 * c, 4 * r, 4),
                                             ("conv stem: a 13x13 receptive field", 4 * c - 6, 4 * r - 6, 13)]):
    ax.imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest", extent=(0, 32, 32, 0))
    ax.add_patch(plt.Rectangle((x0, y0), size, size, fill=False, edgecolor="white", lw=2))
    ax.set_title(f"{title}\n(token at row {r}, column {c})", fontsize=10)
    ax.axis("off")
plt.show()

## 4. The baseline: ResNet-18

The baseline is the CIFAR version of ResNet-18 (He et al., 2016): a 3x3 convolution stem with stride 1 and no max-pool, then 8 residual blocks in 4 stages with 64, 128, 256 and 512 channels (stages 2 to 4 halve the resolution), global average pooling and one linear layer. It has 11.2M parameters, three times as many as ViT-Lite-7, and it was trained with the same data, augmentation and 300 epochs (with SGD, the usual optimizer for ResNets, and its own tuned learning rate).

Its design builds in two assumptions that the ViT has to learn from data: every layer looks only at a 3x3 neighbourhood, and the same filters are applied at every position. The cell runs the image through each stage, checks the result against the full model and shows the prediction.

In [ ]:
resnet = load_model("resnet18")
feats = [("input", x)]
f = resnet.stem(x)
feats.append(("stem (3x3 conv, 64 channels)", f))
for i, block in enumerate(resnet.stages, start=1):
    f = block(f)
    feats.append((f"residual block {i}", f))
pooled = f.mean(dim=(2, 3))                # global average pooling: (1, 512)
logits_rn = resnet.fc(pooled)              # (1, 10)
display(pd.DataFrame([(n, tuple(t.shape)) for n, t in feats] + [("global average pool", tuple(pooled.shape)),
                     ("linear head (logits)", tuple(logits_rn.shape))], columns=["stage", "shape"]).set_index("stage"))
print(f"largest difference from the library: {(logits_rn - resnet(x)).abs().max().item():.1e}")
pr = logits_rn.softmax(-1)[0].cpu()
print(f"ResNet-18 prediction: {CLASSES[int(pr.argmax())]} ({pr.max().item():.1%}); true class: {CLASSES[label]}")
models["resnet18"] = resnet

## 5. Replication anchor and claim check

**One image per class.** First, every model on the ten images shown in section 1.

In [ ]:
names = {"resnet18": "ResNet-18", **vits}
xb = eval_tf(x_test[first_of_class]).to(DEVICE)
table = {"true class": [CLASSES[c] for c in range(10)]}
for run, name in names.items():
    pr = models[run](xb).softmax(-1).cpu()
    table[name] = [f"{CLASSES[int(p.argmax())]} ({p.max().item():.0%})" + ("" if int(p.argmax()) == c else " x")
                   for c, p in enumerate(pr)]
display(pd.DataFrame(table).set_index("true class"))
print("x marks a wrong prediction; the percentage is the model's probability for its prediction.")

**The test set.** Each model now classifies the test images in batches, exactly as during training (bfloat16 autocast on a GPU). On the full test set the accuracy should match the accuracy the training run logged for seed 0, up to rounding from storing the weights in float16.

In [ ]:
def evaluate(model, x_all, y_all, batch_size=500):
    conf = torch.zeros(10, 10, dtype=torch.long)
    for i in range(0, len(y_all), batch_size):
        xb = eval_tf(x_all[i:i + batch_size]).to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=AMP, enabled=AMP is not None):
            pred = model(xb).argmax(-1).cpu()
        conf += torch.bincount(y_all[i:i + batch_size] * 10 + pred, minlength=100).view(10, 10)
    return conf


n_eval = N_EVAL or (len(y_test) if DEVICE.type == "cuda" else 2000)
eval_models = {run: load_model(run, attn="sdpa") for run in names}  # faster attention, same weights
rows, confusions = [], {}
for run, name in names.items():
    conf = evaluate(eval_models[run], x_test[:n_eval], y_test[:n_eval])
    confusions[run] = conf
    acc = conf.diag().sum().item() / conf.sum().item() * 100
    logged = RUNS[run]["final_test_acc"] * 100
    rows.append({"model": name, "test images": n_eval, "accuracy here (%)": acc,
                 "seed-0 run, logged at the last epoch (%)": logged, "difference (points)": acc - logged})
display(pd.DataFrame(rows).set_index("model"))
if n_eval < len(y_test):
    print(f"Only the first {n_eval} test images were used, so the accuracy here can differ from the full-test-set "
          "number. Set N_EVAL = 10000 in the setup cell for the full test set.")

**The pretrained reference, ViT-B/16.** `google/vit-base-patch16-224-in21k`, pretrained on ImageNet-21k and fine-tuned on CIFAR-10 at 224x224 pixels for 5 epochs. Its 86M parameters take 172 MB even in float16, too much for the repository, so it is evaluated only where its training checkpoint exists (on Wulver); elsewhere the cell prints the run's logged result.

In [ ]:
vitb_dir = REPO / "results/vitb16_finetune/seed0"
vitb_metrics = json.loads((vitb_dir / "metrics.json").read_text()) if (vitb_dir / "metrics.json").exists() else None
if (vitb_dir / "last.pt").exists():
    import yaml

    vitb = build_model({"name": "vit_b16_architecture", "attn": "sdpa"})  # same architecture, no download
    vitb.load_state_dict(torch.load(vitb_dir / "last.pt", map_location="cpu", weights_only=False)["model"])
    vitb = vitb.to(DEVICE).eval()
    vitb_tf = build_transform(yaml.safe_load((vitb_dir / "config.yaml").read_text())["data"], train=False)
    xb = vitb_tf(img.unsqueeze(0)).to(DEVICE)
    print("ViT-B/16 input:", tuple(xb.shape), "(the 32x32 image resized to 224x224: 14x14 = 196 patches of 16x16)")
    pr = vitb(xb).softmax(-1)[0].cpu()
    print(f"prediction for test image {IMG}: {CLASSES[int(pr.argmax())]} ({pr.max().item():.1%})")
    conf = torch.zeros(10, 10, dtype=torch.long)
    for i in range(0, n_eval, 250):
        xb = vitb_tf(x_test[i:min(i + 250, n_eval)]).to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=AMP, enabled=AMP is not None):
            pred = vitb(xb).argmax(-1).cpu()
        conf += torch.bincount(y_test[i:min(i + 250, n_eval)] * 10 + pred, minlength=100).view(10, 10)
    print(f"test accuracy on {n_eval} images: {conf.diag().sum().item() / conf.sum().item():.2%} "
          f"(the run logged {vitb_metrics['final_test_acc']:.2%})")
elif vitb_metrics:
    print(f"ViT-B/16 checkpoint not available here; the fine-tuning run logged {vitb_metrics['final_test_acc']:.2%} "
          "test accuracy.")

**Three seeds, the published numbers and the claim.** The report's numbers are the mean and standard deviation over 3 training seeds (0, 1, 2), read here from the run results in `results/`. Two published numbers serve as replication anchors: ViT-Lite-7/8 and ViT-Lite-7/4 from Hassani et al. (2021, 300 epochs), and ViT-B/16 from Dosovitskiy et al. (2021, fine-tuned at 384 px). The claim holds at a patch size if ResNet-18's mean is at least 1 point above the ViT's mean.

In [ ]:
PUBLISHED = {"vitlite_p8": (89.10, "Hassani et al. (2021)"), "vitlite_p4": (93.57, "Hassani et al. (2021)"),
             "vitb16_finetune": (98.95, "Dosovitskiy et al. (2021), 384 px")}
all_names = {**names, "vitb16_finetune": "ViT-B/16, pretrained + fine-tuned"}
seed_acc = {}
for run in all_names:
    files = sorted((REPO / "results" / run).glob("seed*/metrics.json"))
    seed_acc[run] = np.array([json.loads(f.read_text())["final_test_acc"] * 100 for f in files])

rows = []
for run, name in all_names.items():
    a = seed_acc[run]
    pub, source = PUBLISHED.get(run, (None, ""))
    rows.append({"model": name, "seeds": len(a), "test accuracy (%), mean": a.mean(),
                 "std": a.std(ddof=1) if len(a) > 1 else None, "published (%)": pub,
                 "difference (points)": a.mean() - pub if pub else None, "source": source})
display(pd.DataFrame(rows).set_index("model"))

base = seed_acc["resnet18"].mean()
claim = [{"patch size": f"{ps}x{ps}", "ViT-Lite-7 (%)": seed_acc[run].mean(), "ResNet-18 (%)": base,
          "gap (points)": base - seed_acc[run].mean(), "at least 1 point?": "yes" if base - seed_acc[run].mean() >= 1 else "no"}
         for ps, run in [(8, "vitlite_p8"), (4, "vitlite_p4"), (2, "vitlite_p2")]]
display(pd.DataFrame(claim).set_index("patch size"))
verdict = all(row["at least 1 point?"] == "yes" for row in claim)
print("Claim:", "holds at every patch size tested." if verdict else "fails for at least one patch size.")

figure = REPO / "results/figures/claim.png"
if figure.exists():
    from IPython.display import Image
    display(Image(filename=str(figure), width=760))

## 6. Try your own image

`classify` accepts a file path, a PIL image or an (H, W, 3) array. It shrinks the picture to 32x32, the only size these models know, applies the same preparation as in section 1 and prints each model's three most likely classes. The example uses test image 0 flipped left to right, an image no model has seen in exactly this form. On Colab, uncomment the last two lines to upload a photo.

In [ ]:
from PIL import Image as PILImage


def classify(image, top=3):
    if isinstance(image, (str, Path)):
        image = PILImage.open(image)
    if not isinstance(image, PILImage.Image):
        image = PILImage.fromarray(np.asarray(image, dtype=np.uint8))
    small = image.convert("RGB").resize((32, 32), PILImage.BICUBIC)
    xb = eval_tf(torch.from_numpy(np.asarray(small).copy()).permute(2, 0, 1)).unsqueeze(0).to(DEVICE)
    plt.figure(figsize=(1.6, 1.6))
    plt.imshow(small, interpolation="nearest")
    plt.axis("off")
    plt.title("as the models see it", fontsize=9)
    plt.show()
    rows = []
    for run, name in names.items():
        pr = models[run](xb).softmax(-1)[0].cpu()
        best = pr.topk(top)
        rows.append({"model": name, **{f"#{k + 1}": f"{CLASSES[int(i)]} ({v.item():.0%})"
                                       for k, (v, i) in enumerate(zip(best.values, best.indices))}})
    display(pd.DataFrame(rows).set_index("model"))


classify(x_test[IMG].flip(-1).permute(1, 2, 0).numpy())
# from google.colab import files
# classify(next(iter(files.upload())))

## 7. Summary: what goes in, what comes out

| Model | Input | What the model computes | Output |
| --- | --- | --- | --- |
| ViT-Lite-7/8, 7/4, 7/2 | (B, 3, 32, 32) normalised image | Cuts it into 16, 64 or 256 patches, embeds them, adds a CLS token and positions, applies 7 attention + MLP blocks, reads the CLS token | (B, 10) logits |
| ViT-Lite-7 + conv stem | (B, 3, 32, 32) normalised image | Two conv + pool stages make 64 overlapping local tokens; the rest is the same as above | (B, 10) logits |
| ResNet-18 (baseline) | (B, 3, 32, 32) normalised image | 3x3 convolutions in 8 residual blocks, global average pooling | (B, 10) logits |
| ViT-B/16 (pretrained reference) | (B, 3, 224, 224) image resized from 32x32 | 196 patches of 16x16, 12 blocks of width 768, pretrained on ImageNet-21k | (B, 10) logits |

Softmax of the logits gives the class probabilities; the prediction is the most likely class. Sections 2 to 4 checked every hand-computed step against the library models. Section 5 measured the test accuracy of each exported model and set it against the training runs, the published numbers and the claim. The report (`report/`) explains what the layer analyses in `results/analysis/` show about why the from-scratch ViTs stay behind.